# Film Öneri Sistemi

Bu projede izlenen filme benzer filmleri önereceğim. İçerik tabanlı bakacağım.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/MoviesOnStreamingPlatforms_updated.csv')
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


### Görselleştirme


In [ ]:
df['Year'].hist(bins=30)
plt.show()


In [ ]:
df[['Netflix','Hulu','Prime Video','Disney+']].sum().plot(kind='bar')
plt.show()


### Boş veri


In [ ]:
df['IMDb']=df['IMDb'].fillna(df['IMDb'].median())
df['Genres']=df['Genres'].fillna('')
df['Directors']=df['Directors'].fillna('')
df['Runtime']=df['Runtime'].fillna(df['Runtime'].median())


### Feature Engineering


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
df['soup']=df['Genres']+' '+df['Directors']
vec=TfidfVectorizer(max_features=2000)
mat=vec.fit_transform(df['soup'])


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
tr,te=train_test_split(df.index,test_size=0.2,random_state=42)


### 3 yaklaşım


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity,linear_kernel,euclidean_distances
import numpy as np

def oner(i,sim,n=5):
    skor=sim[i]
    idx=np.argsort(skor)[::-1][1:n+1]
    return df.iloc[idx]['Title']

# küçük örnek için ilk 1500 film
M=mat[:1500]
coss=cosine_similarity(M)
lin=linear_kernel(M)
eu=1/(1+euclidean_distances(M))
print(oner(10,coss).tolist())
print(oner(10,lin).tolist())


In [ ]:
import joblib
joblib.dump({'vec':vec,'titles':df['Title'].tolist()},'../../models/recsys_movie.joblib')


### Sonuç

Tür + yönetmen ile benzer film çıkıyor. Cosine daha anlamlı geldi. Hedefi tutturdum.
